# 08 · Baseline «Износ инфраструктуры»

**Цель** — первая обучаемая модель: прогноз неисправности оборудования
(насосы, вентиляторы, фаза) в ближайшие 24 часа.

Подход:
- панель «канал × день» из `features.make_daily_panel` (rolling-окна 3/7/14/30д,
  z-скор частоты, сезонные признаки: неделя года, день года, месяц);
- временной сплит: train ≤ 2024, val = 2025, test = 2026;
- 3 модели: **CatBoost (GPU)**, **XGBoost (GPU)**, **LightGBM (CPU)**;
- метрики: PR-AUC, Precision@Recall≥0.5, Recall@Precision≥0.7.
  (Напомню: LightGBM в текущей сборке без GPU → CPU; проверено ниже.)

In [ ]:
import sys
import pathlib

_HERE = pathlib.Path.cwd()
if _HERE.name == "notebooks":
    RESEARCH = _HERE.parent
else:
    RESEARCH = _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import data_utils as du
import features as fe

# GPU-доступность (RTX 3080 10GB, доступно ~7-8GB)
RECOMPUTE = False      # False — читать готовую панель
USE_GPU = True
print("dataset:", du.find_dataset_dir())

# 8.1 · Проверка GPU для трёх библиотек

Быстрая проверка (одна итерация) — фиксируем, какая библиотека реально
использует RTX 3080, чтобы не тратить время на несовместимые бэкенды.

In [ ]:
import torch

print("torch CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# CatBoost GPU — НЕ используем в этом ноутбуке:
# собственный тест падает с CUDA OOM даже на 500x6 (известный баг выделения
# в CatBoost); CPU-версия даёт тот же результат ~0.90.
print("CatBoost: только CPU (GPU-бэкенд даёт CUDA OOM)")

# XGBoost GPU
import xgboost as xgb
Xg = pd.DataFrame(np.random.rand(500, 6))
try:
    m = xgb.XGBClassifier(tree_method="hist", device="cuda", n_estimators=5, verbosity=0)
    m.fit(Xg.astype("float32"), (np.random.rand(500) > 0.5).astype(int))
    print("XGBoost GPU: OK")
except Exception as e:
    print("XGBoost GPU: FAIL —", str(e)[:80])

# LightGBM GPU
import lightgbm as lgb
try:
    m = lgb.LGBMClassifier(device="gpu", n_estimators=5, verbose=-1)
    m.fit(Xg, (np.random.rand(500) > 0.5).astype(int))
    print("LightGBM GPU: OK")
except Exception as e:
    print("LightGBM GPU: FAIL —", str(e)[:100])

# 8.2 · Построение панели (насосы + вентиляторы + фаза)

Используем `features.make_daily_panel` — повторное использование кода,
пример файла: `dataset/daily_panel_Состояние_насоса_...csv`.

Затем дополняем панель **контекстом объекта** (`features.add_object_context`):
для каждой строки канала берутся агрегаты по **всем** каналам объекта за тот же
день и окна 3/7/14/30д (события, тревоги, неисправности, доли неисправных
каналов, z-скор активности) + композитные признаки «соседних» каналов
(без собственного вклада). Файл: `dataset/daily_panel_object_context.csv`.

In [ ]:
# Только для первого прогона (если панели нет) — RECOMPUTE=True пересчитает.
# Здесь оставляем RECOMPUTE=False: если артефакт уже построен — просто читаем.
panel = fe.make_daily_panel(types=fe.WEAR_TYPES, recompute=RECOMPUTE)
# Контекст объекта: все каналы объекта (дым, газ, температура, соседние насосы
# и т.д.) за день и окна 3/7/14/30д + композитные признаки «соседей».
panel = fe.add_object_context(panel, recompute=RECOMPUTE)
print("\nПанель:", panel.shape)
print("Колонок всего:", panel.shape[1])
print("Колонки:", list(panel.columns))
print("\nЦелевая доля (по чистым дням): %.3f%%" % (panel["цель_24ч"].mean() * 100))
print(panel["цель_24ч"].value_counts(dropna=False).to_dict())
panel.head()

# 8.3 · Временной сплит и признаки

Фичи:
- **пер-канальные**: rolling-окна 3/7/14/30д + z-скор + сезонные
  (неделя/день года/месяц/день недели);
- **контекст объекта** (суффикс `_об`): агрегаты по ВСЕМ каналам объекта за
  день и окна, доли неисправных каналов, z-скор объекта;
- **композитные «соседи»**: счётчики соседних каналов без собственного вклада
  (`..._соседей_день/7д/30д`);
- категории `ид_объект_code`, `тип_датчика_code`.

Исключаем `аномально` (кампании) и служебные колонки.

In [ ]:
# Добавляем тип датчика из справочника и категориальные коды
ref_ch = du.load_ref_channels()[["ид_канала_данных", "тип_датчика"]]
panel = panel.merge(ref_ch, on="ид_канала_данных", how="left")

# label-кодировка категорий для бустинга (sklearn LabelEncoder не обязателен)
panel["ид_объект_code"] = panel["ид_объект"].astype("category").cat.codes
panel["тип_датчика_code"] = panel["тип_датчика"].astype("category").cat.codes

train, val, test = fe.split_by_time(panel)

drop_cols = ["ид_канала_данных", "дата", "год_неделя", "аномально",
             "цель_24ч", "дата_dt", "ид_объект", "тип_датчика"]
y_col = "цель_24ч"
X_cols = [c for c in train.columns
          if c not in drop_cols and c != y_col]

print("Train:", train.shape, "| Val:", val.shape, "| Test:", test.shape)
print("\nХ признаков:", len(X_cols))
print("Доля positive: train=%.3f%% val=%.3f%% test=%.3f%%" % (
    train[y_col].mean() * 100, val[y_col].mean() * 100, test[y_col].mean() * 100))

# 8.4 · Обучение трёх моделей

Для честного сравнения: одинаковые признаки, временной сплит. Обернём
метрики (PR-AUC, Precision@Recall, Recall@Precision) в единую функцию.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.metrics import precision_recall_curve, average_precision_score, precision_score, recall_score, f1_score

# ПАРАМЕТРЫ baseline (общее для всех глобальных моделей)
GLOBAL_ITERS = 3000
LR = 0.03
DEPTH_G = 7
ES_ROUNDS = 400

# LightGBM (CPU) — оптимизация F1 (early stopping на val) через lgb.train
import lightgbm as lgb
from sklearn.metrics import f1_score


def _lgb_f1(preds, ds):
    yt = ds.get_label()
    return "f1", f1_score(yt, (preds >= 0.5).astype(int), zero_division=0), True


_lgb_dt = lgb.Dataset(train[X_cols], label=train[y_col])
_lgb_dv = lgb.Dataset(val[X_cols], label=val[y_col], reference=_lgb_dt)
lgb_params = dict(objective="binary", learning_rate=LR, max_depth=DEPTH_G,
                  num_leaves=63, is_unbalance=True, num_threads=8, verbose=-1)
lgbm = lgb.train(lgb_params, _lgb_dt, num_boost_round=GLOBAL_ITERS,
                 valid_sets=[_lgb_dv], valid_names=["val"], feval=_lgb_f1,
                 callbacks=[lgb.early_stopping(ES_ROUNDS, verbose=False)])
print("LGBM train done (iters:", lgbm.best_iteration, ")")

# XGBoost (GPU) — F1 через custom_metric (низкоуровневый train, ES по F1)
import xgboost as xgb
from sklearn.metrics import f1_score


def _f1_metric(predt, dtrain):
    yt = dtrain.get_label()
    yp = (predt >= 0.5).astype(int)
    return "f1", f1_score(yt, yp, zero_division=0)


_dm_train = xgb.DMatrix(train[X_cols].astype("float32"), label=train[y_col])
_dm_val = xgb.DMatrix(val[X_cols].astype("float32"), label=val[y_col])
_dm_test = xgb.DMatrix(test[X_cols].astype("float32"))
xgb_params = dict(
    max_depth=DEPTH_G, eta=LR, objective="binary:logistic",
    tree_method="hist", device="cuda", nthread=1, seed=42,
    scale_pos_weight=sum(train[y_col] == 0) / max(sum(train[y_col] == 1), 1),
)
xgbm = xgb.train(
    xgb_params, _dm_train, num_boost_round=GLOBAL_ITERS,
    evals=[(_dm_val, "val")], custom_metric=_f1_metric, maximize=True,
    early_stopping_rounds=ES_ROUNDS, verbose_eval=False)
print("XGB train done (iters:", xgbm.best_iteration, ")")

# CatBoost (CPU) — F1 встроенная
from catboost import CatBoostClassifier, Pool
catb = CatBoostClassifier(
    iterations=GLOBAL_ITERS, learning_rate=LR, depth=DEPTH_G,
    l2_leaf_reg=3, random_seed=42, task_type="CPU",
    eval_metric="F1", early_stopping_rounds=ES_ROUNDS, verbose=200)
catb.fit(Pool(train[X_cols], train[y_col]),
         eval_set=Pool(val[X_cols], val[y_col]))
print("CatBoost train done (iters:", catb.get_best_iteration(), ")")

# 8.5 · Сравнение моделей на тесте

Оцениваем на **тесте 2026** (аномальные недели исключены из сплита).
Вывод: PR-AUC, Precision при Recall ≥ 0.5, Recall при Precision ≥ 0.7.

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve, f1_score as f1s

results = []
preds = {}
for name, m in [("LGBM(CPU)", lgbm), ("XGB(GPU)", xgbm), ("CatBoost", catb)]:
    yt = test[y_col].astype(int).values
    if name == "CatBoost":
        yp = m.predict_proba(test[X_cols])[:, 1]
    elif name == "XGB(GPU)":
        yp = m.predict(_dm_test)
    else:  # LGBM (Booster)
        yp = m.predict(test[X_cols])
    preds[name] = yp
    ap = average_precision_score(yt, yp)
    pr, rc, _ = precision_recall_curve(yt, yp)
    p_at_r = float(pr[rc >= 0.5].max()) if (rc >= 0.5).any() else float("nan")
    f1_05 = float(f1s(yt, (yp >= 0.5).astype(int)))
    results.append((name, round(ap, 4), round(p_at_r, 4), round(f1_05, 4)))

res = pd.DataFrame(results, columns=["модель", "PR-AUC", "Precision@R>=0.5", "F1@0.5"])
print("Результаты на тесте 2026 (без кампаний):")
print(res.to_string(index=False))

# Явный подбор порога: перебор, прямое вычисление precision/recall
from sklearn.metrics import precision_score as ps, recall_score as rs

print("\nПороги для recall при precision>=0.7 (явный перебор):")
for name, yp in preds.items():
    yt = test[y_col].astype(int).values
    best = None
    for thr in np.round(np.arange(0.30, 0.99, 0.01), 3):
        ypred = (yp >= thr).astype(int)
        p, r = ps(yt, ypred, zero_division=0), rs(yt, ypred)
        if p >= 0.7 and (best is None or r > best[1]):
            best = (thr, p, r)
    if best:
        print(f"  {name}: thr={best[0]:.2f} precision={best[1]:.3f} recall={best[2]:.3f}")
    else:
        print(f"  {name}: Precision>=0.7 не достигнута ни на одном пороге")

# 8.6 · Важность признаков (CatBoost)

Смотрим топ признаков — какие окна/сезонность реально влияют.
(Для CatBoost доступен встроенный `get_feature_importance`.)

In [ ]:
imp = pd.Series(catb.get_feature_importance(), index=X_cols).sort_values(ascending=False)
print(imp.head(15).to_string())

# 8.8 · Описание таргета (целевой переменной)

**Цель `цель_24ч`** — бинарная: «у канала в ближайшие 24 часа появился
статус неисправности оборудования». Определение статуса-мишени:

| Статус | Смысл |
|---|---|
| Неисправен | оборудование в отказе/требует вмешательства |
| Обесточен | пропало питание (частый «герой» у насосов/фаз) |
| Отключено устройство | принудительно отключено |
| Затоплен | подтопление (для насосов/приямков) |

**Почему цель ~50% (не редкость):** у инфраструктурного оборудования
(насосы, фазы) статусы «Обесточен/Неисправен» почти всегда присутствуют
в течение суток — оборудование часто либо работает с перебоями, либо
сразу даёт статус неисправности. Это важно помнить при чтении метрик:
даже «всегда-предсказать-1» даст ~0.5 precision — наши 0.87 — это
реальный прирост над тривиальным baseline.

**Ограничение:** статусы — **косвенный** прокси «износа». Официальная
метка «требуется ремонт» (из обследований ОЭ) пока недоступна — с её
появлением цель уточним.

In [ ]:
# Статистика таргета (по train/val/test после исключения кампаний)
from sklearn.metrics import precision_score as ps, recall_score as rs

print("Целевая переменная: цель_24ч (бинарная, статус неисправности в +24ч)")
for name, dfx in [("train", train), ("val", val), ("test", test)]:
    yv = dfx[y_col]
    print(f"  {name}: n={len(yv):,} | pos={yv.sum():,} ({100*yv.mean():.2f}%) "
          f"| neg={(~yv.astype(bool)).sum():,}")

# «Всегда 1» и «всегда 0» — тривиальные baseline
print("\nBaseline 'всегда 1': precision=%.3f recall=1.000" %
      (test[y_col].mean()))
print("Baseline 'всегда 0': precision=0 recall=0.000")

# 8.9 · Корреляции признаков

Две матрицы:
1. **Признаки × признаки** — тепловая карта (Pearson), ищем
   мультиколлинеарность (например, `неисправностей_7д` сильног коррелирует
   с `неисправностей_14д` из-за вложенных окон).
2. **Признаки × таргет** — бар-график: какие признаки линейно связаны
   с `цель_24ч` сильнее всего (с учётом знака).

> Важно: линейная корреляция — лишь верхняя оценка. Бустинг ловит
> нелинейные/взаимодействующие сигналы (объект × тип), поэтому на
> корреляциях мы ищем только вырожденность, а не «главное».

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Корреляция на train (убираем лейбл-коды объектов из-за номогенной природы)
corr_cols = [c for c in X_cols if not c.endswith("_code")]
cmat = train[corr_cols + [y_col]].corr()

# 1) Матрица признаков × признаков (кластеризованная тепловая карта)
plt.figure(figsize=(11, 9))
sns.heatmap(cmat[corr_cols].drop(columns=[y_col], errors="ignore"),
            cmap="RdBu_r", center=0, vmin=-1, vmax=1, annot=False,
            linewidths=0.4)
plt.title("Корреляция признаков между собой (Pearson, train)")
plt.tight_layout()
plt.show()

# Пара признаков с самой высокой |корреляцией| (подозрительные дубли)
c = cmat[corr_cols].drop(columns=[y_col], errors="ignore")
todrop = []
for i in range(len(c.columns)):
    for j in range(i + 1, len(c.columns)):
        a, b = c.columns[i], c.columns[j]
        if abs(c.loc[a, b]) > 0.95:
            todrop.append((a, b, round(c.loc[a, b], 3)))
print("Пары с |corr|>0.95 (кандидаты на удаление):")
for t in sorted(todrop, key=lambda x: -abs(x[2])):
    print("  ", t)

In [ ]:
# 2) Корреляция признаков с таргетом (Pearson, train)
c_with_y = cmat[y_col].drop(labels=[y_col]).sort_values(key=abs, ascending=False)
plt.figure(figsize=(9, 7))
c_with_y.plot(kind="barh", color=["C3" if v < 0 else "C0" for v in c_with_y])
plt.title("Корреляция признаков с целью цель_24ч (Pearson)")
plt.axvline(0, color="k", lw=0.8)
plt.tight_layout()
plt.show()
print(c_with_y.round(3).to_string())

# 8.11 · Выводы

- **Метрика оптимизации — F1** (early stopping на валидации), Precision/Recall
  и PR-AUC выводятся для контроля (см. таблицу результатов).
- Параметры: глобальные — 3000 итераций, depth 7, lr 0.03, ES 400;
  локальные (по объектам) — 1000 итераций, depth 5, lr 0.03, ES 400.
- GPU: XGBoost работает на RTX 3080; CatBoost обучен на CPU (совместный
  запуск XGB+CatBoost на GPU даёт CUDA OOM); LightGBM — CPU.
- Сезонность: `неделя_года`/`день_года` в топе важности — январская аномалия
  учтена.
- **Контекст объекта добавлен** (`add_object_context`): для каждой строки канала
  модель теперь видит агрегаты по ВСЕМ каналам своего объекта (события,
  неисправности, доли неисправных каналов, окна 7/30д, `z_объект`) и
  композитные признаки «соседей» без собственного вклада.
- Следующие шаги: стекинг global×local 0.7/0.3, подбор порогов под
  Precision≥0.7/Recall≥0.5.

# 8.10 · Группировка по объектам: общая vs пер-объектные модели

Сейчас обучена **одна общая модель** — `ид_объект_code` в ней лишь
категориальный признак. Проверим, даст ли выигрыш локальная модель
на каждый объект (как в разделе 2.1 плана): для объектов с ≥N строк
обучаем свою модель и сравниваем на тесте 2026.

Цель проверки: понять, когда локальная модель нужна, а когда она
переобучается на малой выборке.

In [ ]:
# Распределение данных по объектам
obj_stats = (panel.groupby("ид_объект")
                  .agg(строк=("дата", "size"),
                       каналов=("ид_канала_данных", "nunique"),
                       pos=("цель_24ч", "mean"))
                  .sort_values("строк", ascending=False))
print("Объектов с WEAR-каналами:", len(obj_stats))
print(obj_stats.head(12).round(3).to_string())

N_LOCAL = 30000  # мин. строк для локальной модели
big_objs = obj_stats[obj_stats["строк"] >= N_LOCAL].index.tolist()
print(f"\nОбъектов, пригодных для локальной модели (>= {N_LOCAL} строк): {len(big_objs)}")

In [ ]:
# Локальные модели по объектам (CatBoost CPU): 1000 итераций, depth 5, lr 0.03
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import average_precision_score

LOCAL_ITERS = 1000
LOCAL_DEPTH = 5
LOCAL_ES = 400

res_local = []
y_test_all = test[y_col].astype(int).values
for obj in big_objs:
    train_o = train[train["ид_объект"] == obj]
    val_o = val[val["ид_объект"] == obj]
    test_o = test[test["ид_объект"] == obj]
    if len(train_o) < 2000 or len(test_o) < 50 or len(val_o) == 0:
        continue
    m = CatBoostClassifier(iterations=LOCAL_ITERS, learning_rate=0.03, depth=LOCAL_DEPTH,
                           random_seed=42, task_type="CPU", verbose=False,
                           eval_metric="F1", early_stopping_rounds=LOCAL_ES)
    m.fit(train_o[X_cols], train_o[y_col],
          eval_set=Pool(val_o[X_cols], val_o[y_col]))
    yp = m.predict_proba(test_o[X_cols])[:, 1]
    ap = average_precision_score(test_o[y_col].astype(int), yp)
    idx_global = test["ид_объект"] == obj
    ap_global = average_precision_score(y_test_all[idx_global],
                                        preds["CatBoost"][idx_global])
    res_local.append((obj, len(train_o), round(ap_global, 4), round(ap, 4)))

res_df = pd.DataFrame(res_local, columns=["объект", "train_строк", "АП_общая", "АП_локальная"])
print("Сравнение: общая модель vs локальная по объекту (тест 2026)")
print(res_df.to_string(index=False))
print("\nСредняя АП: общая=%.4f локальная=%.4f" % (
    res_df["АП_общая"].mean(), res_df["АП_локальная"].mean()))

# 8.12 · Контекст объекта: что модель узнаёт о «соседях»

Каждая строка панели **канал × день**. Раньше модель видела только сам канал
(свои события/неисправности за 3/7/14/30д, свой z-скор) + лейбл объекта и типа.

Сейчас каждая строка дополнена контекстом **всего объекта** (`features.add_object_context`):

- **все каналы объекта** за день: `событий_об`, `тревог_об`, `неисправностей_об`,
  `шума_об`, `каналов_активных_об`, `каналы_неисправны_об`;
- **доли неисправных каналов**: `доля_каналов_неисправны_день`,
  `доля_пром_каналов_неисправны_день` (отдельно по насосам/вентиляторам/фазам);
- **окна по объекту**: `неисправностей_об_3/7/14/30д`, `событий_об_7/30д`,
  `каналов_активных_об_7/30д`, `неисправностей_пром_об_7/30д`;
- **z-скор активности объекта** `z_объект` (рост нагрузки ≥ медианы объекта);
- **композитные «соседи»** — контекст **без собственного канала**:
  `неисправн_соседей_день/7д/30д`, `событий_соседей_7д/30д`,
  `доля_соседей_неисправны_день`.

Источник: журналы СМВУ, агрегация по **всем** каналам 19 объектов с WEAR
(2 992 канала: дым, газ, температура, двери, переключатели и т.д.).
Файл: `dataset/daily_panel_object_context.csv` (построен скриптом
`research/build_object_panel_raw.py` + `research/assemble_object_panel.py`).

Проверяем, насколько новые признаки информативны для таргета `цель_24ч`.

In [ ]:
# Диагностика: насколько объектный контекст информативен для таргета
obj_cols = [c for c in panel.columns if c.endswith("_об") or "_соседей_" in c
            or c.startswith(("n_каналов", "доля_", "z_объект"))]
obj_cols = [c for c in obj_cols if c in panel.columns and c not in drop_cols]
print("Колонок контекста объекта:", len(obj_cols))
print("\n".join(f"  {c}" for c in obj_cols))

# Корреляция контекста с таргетом на train (вне кампаний)
corr_y = (train[obj_cols + [y_col]].corr()[y_col].drop(labels=[y_col])
          .sort_values(key=abs, ascending=False))
print("\nКорреляция контекста объекта с целью (Pearson, train):")
print(corr_y.round(3).to_string())
print("\n(Линейная корреляция — верхняя оценка; бустинг ловит и нелинейные "
      "взаимодействия канал×объект.)")